# Dataframer: Robert Fagles's Odyssey to Pandas DF

### [—————————————pipeline—————————————]
### »——raw—»—clean—»—normalize—»—DATAFRAME——»

Here are some transformation and frequencies for future exploratory analysis of Green's Odyssey.

Columns: author, year, title, book_num, text, num_lines, num_sentences, num_words, 


In [1]:
# library imports
import os

import numpy as np
import pandas as pd

import re
import nltk

In [2]:
# Display options
pd.set_option("display.max_colwidth", None)  # Prevent truncation of long values
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.expand_frame_repr", False)  # Prevent wrapping in DataFrames

In [3]:
# Visualization libraries
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/Users/debr/English-Homer') 
import bard_visualization as viz# This will apply the visualization settings
from bard_visualization import color_palette 
sys.path.append('/Users/debr/English-Homer/functions') 
import e_nlp as e

Functions for NLP are live! use e.<function> to call them.
Download complete.
Stopwords customized:
  Added: {'eight', 'ten', 'six', 'two', "'and", "'", 'seven', 'three', 'one', 'four', 'n', 'nine', 'five'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'-', '’', '—', '”', '\\', '‘', '“', '…'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# TO UPDATE
translator = "Green"
book_breaker = "Book"

# Check Paths
filepath = f"/Users/debr/odysseys_en/Normalized_txts/Odyssey_{translator}_Normalized.txt"
output_path = f"/Users/debr/English-Homer/dataframers_by_author/{translator}_DFed/"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_path_plots = f"{output_path}/plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)

# READING FILE TO extracted_lines
with open(filepath, 'r') as file:
    extracted_lines = file.readlines()

text = "".join(extracted_lines)

In [5]:
# Function to split the text into books

#books = string_into_books(text, book_breaker)
books = e.list_into_books(extracted_lines, book_breaker)

# Verify the results
print(f"Found {len(books)} books")
for i, book in enumerate(books):
    print(f"Book {i+1} has {len(book)} lines")
    print(f"Book {i+1} starts with: {book[0]}")

Found 24 books
Book 1 has 446 lines
Book 1 starts with: The man, Muse—tell me about that resourceful man, who wandered

Book 2 has 434 lines
Book 2 starts with: When Dawn appeared, early risen and rosy-fingered,

Book 3 has 497 lines
Book 3 starts with: Deserting the deep’s enchanting surface, the sun rose up

Book 4 has 850 lines
Book 4 starts with: Now they came to deep-hollowed Lakedaimon with its ravines

Book 5 has 495 lines
Book 5 starts with: As Dawn arose from her bed beside illustrious Tithonos,

Book 6 has 331 lines
Book 6 starts with: So Odysseus slept on there, godlike and much-enduring,

Book 7 has 349 lines
Book 7 starts with: So while Odysseus prayed there, godlike and much-enduring,

Book 8 has 589 lines
Book 8 starts with: When Dawn appeared, early risen and rosy-fingered,

Book 9 has 570 lines
Book 9 starts with: Then resourceful Odysseus responded to him, saying:

Book 10 has 578 lines
Book 10 starts with: “To the isle of Aiolia then we came, where was the dwelling



In [6]:
# Books (lists) into DataFrame
df = e.book_into_df(f"{translator}", "2018", "The Odyssey", books)

# Apply functions & add new columns
df['num_lines'] = df['text'].apply(e.count_lines)
df['num_sentences'] = df['text'].apply(e.count_sentences)
df['num_words'] = df['text'].apply(e.count_words)

In [7]:
# Initialize the pipeline
nlp = e.NLPPipeline(language='english')

# Customize stopwords
nlp.customize_stopwords(
    include={'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
             "'", 'n', "'and",},
    exclude={''}
)
# Customize punctuation
nlp.customize_punctuation(
    keep={'-', ""},  # Keep hyphens and apostrophes
    remove={r'…', '—','”','’','“', '‘', '-', '\\'}  # Additional characters to remove
)

# Process with default pipeline (lowercase -> tokenize -> remove punctuation -> remove stopwords)
df = nlp.process_dataframe(df, 'text', 'tokens')
df['num_tokens'] = df['tokens'].map(len)

Stopwords customized:
  Added: {'eight', 'ten', 'two', 'six', "'and", "'", 'seven', 'three', 'one', 'four', 'n', 'nine', 'five'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'-', '’', '—', '”', '\\', '‘', '“', '…'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


In [8]:
print('Type:', type(df['text'][0]))
print('Lenght:',len(df['text'][0]))
print(df['text'][:6])

Type: <class 'list'>
Lenght: 446
0    [The man, Muse—tell me about that resourceful man, who wandered\n, far and wide, when he’d sacked Troy’s sacred citadel:\n, many men’s townships he saw, and learned their ways of thinking,\n, many the griefs he suffered at heart on the open sea,\n, battling for his own life and his comrades’ homecoming. Yet\n, no way could he save his comrades, much though he longed to—\n, it was through their own blind recklessness that they perished,\n, the fools, for they slaughtered the cattle of Helios the sun god\n, and ate them: for that he took from them their day of returning.\n, Tell us this tale, goddess, child of Zeus; start anywhere in it!\n, Now the rest, all those who’d escaped from sheer destruction,\n, were home by now, survivors of both warfare and the sea;\n, Him alone, though longing for his homecoming and his wife,\n, the queenly nymph Kalypso, bright among goddesses,\n, held back in her hollow cavern, desiring him for her husband.\n, But when 

In [9]:
print('Type:', type(df['tokens'][0]))
print('Lenght:',len(df['tokens'][0]))
print(df['tokens'][:6])

Type: <class 'list'>
Lenght: 2160
0                                                                         [man, muse, tell, resourceful, man, wandered, far, wide, sacked, troy, sacred, citadel, many, men, townships, saw, learned, ways, thinking, many, griefs, suffered, heart, open, sea, battling, life, comrades, homecoming, yet, way, could, save, comrades, much, though, longed, blind, recklessness, perished, fools, slaughtered, cattle, helios, sun, god, ate, took, day, returning, tell, us, tale, goddess, child, zeus, start, anywhere, rest, escaped, sheer, destruction, home, survivors, warfare, sea, alone, though, longing, homecoming, wife, queenly, nymph, kalypso, bright, among, goddesses, held, back, hollow, cavern, desiring, husband, year, arrived, circling, seasons, gods, ordained, make, homeward, journey, ithake, even, would, free, trials, even, among, people, ...]
1                                                       [dawn, appeared, early, risen, rosyfingered, odysseus, dear,

In [10]:
#Boolean check for missing values
e.check_df(df)

No missing values

df columns: Index(['author', 'year', 'title', 'book_num', 'text', 'num_lines',
       'num_sentences', 'num_words', 'tokens', 'num_tokens'],
      dtype='object') 

Shape: (24, 10)


In [11]:
df

author  year        title  book_num                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [12]:
# Create output directory if it doesn't exist
output_filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{translator}_DataFrame.csv"
os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

# save df to csv
df.to_csv(output_filepath, index=False)

print(f"Normalization complete. File saved to: {output_filepath}")

Normalization complete. File saved to: /Users/debr/odysseys_en/dataframed/Odyssey_Green_DataFrame.csv
